In [ ]:
!pip3 install googletrans==3.1.0a0

     |████████████████████████████████| 55 kB 1.7 MB/s 
     |████████████████████████████████| 1.3 MB 9.5 MB/s 
     |████████████████████████████████| 42 kB 1.2 MB/s 
     |████████████████████████████████| 53 kB 1.8 MB/s 
     |████████████████████████████████| 65 kB 3.5 MB/s 
  Created wheel for googletrans: filename=googletrans-3.1.0a0-py3-none-any.whl size=16367 sha256=9e591d049e8a3aaadb5e7c52f5704019db15777de101ff94e328dbadd6f81142
  Stored in directory: /root/.cache/pip/wheels/0c/be/fe/93a6a40ffe386e16089e44dad9018ebab9dc4cb9eb7eab65ae
Successfully built googletrans


In [ ]:
pip install google_trans_new

In [ ]:
pip install googletrans


**Membaca Data**

In [ ]:
import tweepy
from textblob import TextBlob
from wordcloud import WordCloud
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
from google.colab import drive
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [ ]:
data=pd.read_csv('/content/medictreat1.csv')
data

,date,username,fulltext
0,2022-03-08 13:17:06,taeforeheaddict,ini dokternya ngasih gue obat apa ya kenapa gu...
1,2022-03-08 13:11:34,nadyaodynstn,@kubiarkankosong Oia betul yg dokternya di tus...
2,2022-03-08 13:11:25,Amaris544572201,@ikatrambutandin Banyak ternyata dokternya..
3,2022-03-08 13:09:28,kyuu_23,@Reeireirin ...bayangin cuma batuk pilek terus...
4,2022-03-08 13:08:19,qoungeun,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...
...,...,...,...
1495,2022-03-04 11:45:23,adri_surcen,OK post dokternya pada cuti 4 hari: meledak
1496,2022-03-04 11:44:30,dprvenus,RT @kukangku: Dokternya bukan kaleng kaleng lo...
1497,2022-03-04 11:42:10,melkkyyway,tp gpp dokternya ganteng😽
1498,2022-03-04 11:39:20,gathaul,@aacronism Hdu berat ya gada dokternya kalo yg...


**Menghapus Data Column yang tidak perlu**

In [ ]:
data.drop(['date','username'], axis=1, inplace=True)
data.rename(columns={'fulltext':'Tweets'})

,Tweets
0,ini dokternya ngasih gue obat apa ya kenapa gu...
1,@kubiarkankosong Oia betul yg dokternya di tus...
2,@ikatrambutandin Banyak ternyata dokternya..
3,@Reeireirin ...bayangin cuma batuk pilek terus...
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...
...,...
1495,OK post dokternya pada cuti 4 hari: meledak
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...
1497,tp gpp dokternya ganteng😽
1498,@aacronism Hdu berat ya gada dokternya kalo yg...


**Preprocessing Cleaning**

Menghapus nama user di kalimat teks

In [ ]:
# remove user
def remove_user(input_txt, pattern):
    r = re.findall(pattern, input_txt)
    for i in r:
        input_txt = re.sub(i, '', input_txt)
    return input_txt    
data['tweet_remove_user'] = np.vectorize(remove_user)(data['fulltext'], "@[\w]*")
data

,fulltext,tweet_remove_user
0,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...
1,@kubiarkankosong Oia betul yg dokternya di tus...,Oia betul yg dokternya di tusuk ya?
2,@ikatrambutandin Banyak ternyata dokternya..,Banyak ternyata dokternya..
3,@Reeireirin ...bayangin cuma batuk pilek terus...,...bayangin cuma batuk pilek terus dokternya ...
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...
...,...,...
1495,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti 4 hari: meledak
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...,"RT : Dokternya bukan kaleng kaleng loh, satu d..."
1497,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng😽
1498,@aacronism Hdu berat ya gada dokternya kalo yg...,Hdu berat ya gada dokternya kalo yg ini


Menghapus yang tidak perlu seperti hastag,emoji,dll

In [ ]:
pip install Sastrawi

     |████████████████████████████████| 209 kB 4.4 MB/s 


In [ ]:
import pandas as pd
import numpy as np
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import re
import unicodedata

In [ ]:
# cleaning
def cleaning(strTweet):
    #remove non-ascii
    strTweet = unicodedata.normalize('NFKD', strTweet).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    #remove URLs
    strTweet = re.sub(r'(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:\'".,<>?«»“”‘’]))', '', strTweet)
    
    #remove punctuations
    strTweet = re.sub(r'[^\w]|_',' ',strTweet)
    
    #remove digit from string
    strTweet = re.sub("\S*\d\S*", "", strTweet).strip()
    
    #remove digit or numbers
    strTweet = re.sub(r"\b\d+\b", " ", strTweet)
    
    #Remove additional white spaces
    strTweet = re.sub('[\s]+', ' ', strTweet)
    
    #remove rt
    strTweet = re.sub(r'^RT[\s]+', '', strTweet)
    
    #remove tab, new line, and backslice
    strTweet = strTweet.replace('\\t', " ").replace('\\n', " ").replace('\\u', " ").replace('\\', "")
    
    # remove whitespace leading & trailing
    strTweet = strTweet.strip()
    
    # remove multiple whitespace into single whitespace
    strTweet = re.sub('\s+',' ',strTweet)
    
    # remove single character
    strTweet = re.sub(r"\b[a-zA-Z]\b", "", strTweet)
    
    return strTweet
data['tweet_cleaning'] = data['tweet_remove_user'].apply(cleaning)
data

,fulltext,tweet_remove_user,tweet_cleaning
0,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...
1,@kubiarkankosong Oia betul yg dokternya di tus...,Oia betul yg dokternya di tusuk ya?,Oia betul yg dokternya di tusuk ya
2,@ikatrambutandin Banyak ternyata dokternya..,Banyak ternyata dokternya..,Banyak ternyata dokternya
3,@Reeireirin ...bayangin cuma batuk pilek terus...,...bayangin cuma batuk pilek terus dokternya ...,bayangin cuma batuk pilek terus dokternya gitu
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA
...,...,...,...
1495,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti hari meledak
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...,"RT : Dokternya bukan kaleng kaleng loh, satu d...",Dokternya bukan kaleng kaleng loh satu dekade ...
1497,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng
1498,@aacronism Hdu berat ya gada dokternya kalo yg...,Hdu berat ya gada dokternya kalo yg ini,Hdu berat ya gada dokternya kalo yg ini


Tahap Case Folding

In [ ]:
# casefolding
def caseFolding(s):
    newStrTweetCaseFold = s.lower()
    
    return newStrTweetCaseFold
data['tweet_case_folding'] = data['tweet_cleaning'].apply(caseFolding)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding
0,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...
1,@kubiarkankosong Oia betul yg dokternya di tus...,Oia betul yg dokternya di tusuk ya?,Oia betul yg dokternya di tusuk ya,oia betul yg dokternya di tusuk ya
2,@ikatrambutandin Banyak ternyata dokternya..,Banyak ternyata dokternya..,Banyak ternyata dokternya,banyak ternyata dokternya
3,@Reeireirin ...bayangin cuma batuk pilek terus...,...bayangin cuma batuk pilek terus dokternya ...,bayangin cuma batuk pilek terus dokternya gitu,bayangin cuma batuk pilek terus dokternya gitu
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA,belum habis kan dokternya
...,...,...,...,...
1495,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti hari meledak,ok post dokternya pada cuti hari meledak
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...,"RT : Dokternya bukan kaleng kaleng loh, satu d...",Dokternya bukan kaleng kaleng loh satu dekade ...,dokternya bukan kaleng kaleng loh satu dekade ...
1497,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng,tp gpp dokternya ganteng
1498,@aacronism Hdu berat ya gada dokternya kalo yg...,Hdu berat ya gada dokternya kalo yg ini,Hdu berat ya gada dokternya kalo yg ini,hdu berat ya gada dokternya kalo yg ini


Tahap Tokenizing

In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# tokenizing
def tweetTokenization(s):
    tokens = word_tokenize(s)

    return tokens
data['tweet_tokenization'] = data['tweet_case_folding'].apply(tweetTokenization)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization
0,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,"[ini, dokternya, ngasih, gue, obat, apa, ya, k..."
1,@kubiarkankosong Oia betul yg dokternya di tus...,Oia betul yg dokternya di tusuk ya?,Oia betul yg dokternya di tusuk ya,oia betul yg dokternya di tusuk ya,"[oia, betul, yg, dokternya, di, tusuk, ya]"
2,@ikatrambutandin Banyak ternyata dokternya..,Banyak ternyata dokternya..,Banyak ternyata dokternya,banyak ternyata dokternya,"[banyak, ternyata, dokternya]"
3,@Reeireirin ...bayangin cuma batuk pilek terus...,...bayangin cuma batuk pilek terus dokternya ...,bayangin cuma batuk pilek terus dokternya gitu,bayangin cuma batuk pilek terus dokternya gitu,"[bayangin, cuma, batuk, pilek, terus, dokterny..."
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA,belum habis kan dokternya,"[belum, habis, kan, dokternya]"
...,...,...,...,...,...
1495,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti hari meledak,ok post dokternya pada cuti hari meledak,"[ok, post, dokternya, pada, cuti, hari, meledak]"
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...,"RT : Dokternya bukan kaleng kaleng loh, satu d...",Dokternya bukan kaleng kaleng loh satu dekade ...,dokternya bukan kaleng kaleng loh satu dekade ...,"[dokternya, bukan, kaleng, kaleng, loh, satu, ..."
1497,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng,tp gpp dokternya ganteng,"[tp, gpp, dokternya, ganteng]"
1498,@aacronism Hdu berat ya gada dokternya kalo yg...,Hdu berat ya gada dokternya kalo yg ini,Hdu berat ya gada dokternya kalo yg ini,hdu berat ya gada dokternya kalo yg ini,"[hdu, berat, ya, gada, dokternya, kalo, yg, ini]"


Tahap Stopwords

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# stopwords
list_stopwords = stopwords.words('indonesian')

list_stopwords.extend(["dg", "ny","d", "klo",
                      "amp", "krn", "n", 'u', 'jd', "nyg",
                       "hehe", "nder", "der", "pen", "sis", "jg",
                       "bgt", 'dah', 'ni', 'so', 'x', 'ri', 'dos', 'eee',
                       'skrng', 'skr', 'kpd', 'j', 's', 'b', 'jgn2', 'gara2',
                       'utk', 'y', 'g', 'm', 'pm', 't', 'dm', 'rm', 'p', 'indonesi', 'https',
                       'ampe', 'rt'
                      ])

list_stopwords = set(list_stopwords)

def removeStopwords(words):
    return ' '.join([word for word in words if word not in list_stopwords])
data['tweet_stopwords'] = data['tweet_tokenization'].apply(removeStopwords)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization,tweet_stopwords
0,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,"[ini, dokternya, ngasih, gue, obat, apa, ya, k...",dokternya ngasih gue obat ya gue keringetan mu...
1,@kubiarkankosong Oia betul yg dokternya di tus...,Oia betul yg dokternya di tusuk ya?,Oia betul yg dokternya di tusuk ya,oia betul yg dokternya di tusuk ya,"[oia, betul, yg, dokternya, di, tusuk, ya]",oia yg dokternya tusuk ya
2,@ikatrambutandin Banyak ternyata dokternya..,Banyak ternyata dokternya..,Banyak ternyata dokternya,banyak ternyata dokternya,"[banyak, ternyata, dokternya]",dokternya
3,@Reeireirin ...bayangin cuma batuk pilek terus...,...bayangin cuma batuk pilek terus dokternya ...,bayangin cuma batuk pilek terus dokternya gitu,bayangin cuma batuk pilek terus dokternya gitu,"[bayangin, cuma, batuk, pilek, terus, dokterny...",bayangin batuk pilek dokternya gitu
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA,belum habis kan dokternya,"[belum, habis, kan, dokternya]",habis dokternya
...,...,...,...,...,...,...
1495,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti hari meledak,ok post dokternya pada cuti hari meledak,"[ok, post, dokternya, pada, cuti, hari, meledak]",ok post dokternya cuti meledak
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...,"RT : Dokternya bukan kaleng kaleng loh, satu d...",Dokternya bukan kaleng kaleng loh satu dekade ...,dokternya bukan kaleng kaleng loh satu dekade ...,"[dokternya, bukan, kaleng, kaleng, loh, satu, ...",dokternya kaleng kaleng loh dekade mendedikasi...
1497,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng,tp gpp dokternya ganteng,"[tp, gpp, dokternya, ganteng]",tp gpp dokternya ganteng
1498,@aacronism Hdu berat ya gada dokternya kalo yg...,Hdu berat ya gada dokternya kalo yg ini,Hdu berat ya gada dokternya kalo yg ini,hdu berat ya gada dokternya kalo yg ini,"[hdu, berat, ya, gada, dokternya, kalo, yg, ini]",hdu berat ya gada dokternya kalo yg


Take the Data after doing Pre-processing

In [ ]:
data['tweet_cleans'] = data['tweet_stopwords']
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization,tweet_stopwords,tweet_cleans
0,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,ini dokternya ngasih gue obat apa ya kenapa gu...,"[ini, dokternya, ngasih, gue, obat, apa, ya, k...",dokternya ngasih gue obat ya gue keringetan mu...,dokternya ngasih gue obat ya gue keringetan mu...
1,@kubiarkankosong Oia betul yg dokternya di tus...,Oia betul yg dokternya di tusuk ya?,Oia betul yg dokternya di tusuk ya,oia betul yg dokternya di tusuk ya,"[oia, betul, yg, dokternya, di, tusuk, ya]",oia yg dokternya tusuk ya,oia yg dokternya tusuk ya
2,@ikatrambutandin Banyak ternyata dokternya..,Banyak ternyata dokternya..,Banyak ternyata dokternya,banyak ternyata dokternya,"[banyak, ternyata, dokternya]",dokternya,dokternya
3,@Reeireirin ...bayangin cuma batuk pilek terus...,...bayangin cuma batuk pilek terus dokternya ...,bayangin cuma batuk pilek terus dokternya gitu,bayangin cuma batuk pilek terus dokternya gitu,"[bayangin, cuma, batuk, pilek, terus, dokterny...",bayangin batuk pilek dokternya gitu,bayangin batuk pilek dokternya gitu
4,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA… https://t.co/kse2yc...,BELUM HABIS KAN DOKTERNYA,belum habis kan dokternya,"[belum, habis, kan, dokternya]",habis dokternya,habis dokternya
...,...,...,...,...,...,...,...
1495,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti 4 hari: meledak,OK post dokternya pada cuti hari meledak,ok post dokternya pada cuti hari meledak,"[ok, post, dokternya, pada, cuti, hari, meledak]",ok post dokternya cuti meledak,ok post dokternya cuti meledak
1496,RT @kukangku: Dokternya bukan kaleng kaleng lo...,"RT : Dokternya bukan kaleng kaleng loh, satu d...",Dokternya bukan kaleng kaleng loh satu dekade ...,dokternya bukan kaleng kaleng loh satu dekade ...,"[dokternya, bukan, kaleng, kaleng, loh, satu, ...",dokternya kaleng kaleng loh dekade mendedikasi...,dokternya kaleng kaleng loh dekade mendedikasi...
1497,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng😽,tp gpp dokternya ganteng,tp gpp dokternya ganteng,"[tp, gpp, dokternya, ganteng]",tp gpp dokternya ganteng,tp gpp dokternya ganteng
1498,@aacronism Hdu berat ya gada dokternya kalo yg...,Hdu berat ya gada dokternya kalo yg ini,Hdu berat ya gada dokternya kalo yg ini,hdu berat ya gada dokternya kalo yg ini,"[hdu, berat, ya, gada, dokternya, kalo, yg, ini]",hdu berat ya gada dokternya kalo yg,hdu berat ya gada dokternya kalo yg


Delete Column yang tidak diperlukan

In [ ]:
data.drop(data.columns[[0,1,2,3,4,5]], axis = 1, inplace = True)

In [ ]:
data

,tweet_cleans
0,dokternya ngasih gue obat ya gue keringetan mu...
1,oia yg dokternya tusuk ya
2,dokternya
3,bayangin batuk pilek dokternya gitu
4,habis dokternya
...,...
1495,ok post dokternya cuti meledak
1496,dokternya kaleng kaleng loh dekade mendedikasi...
1497,tp gpp dokternya ganteng
1498,hdu berat ya gada dokternya kalo yg


Menghapus Data yang Duplikat

In [ ]:
data.drop_duplicates(subset = "tweet_cleans", keep = "first", inplace=True)
data

,tweet_cleans
0,dokternya ngasih gue obat ya gue keringetan mu...
1,oia yg dokternya tusuk ya
2,dokternya
3,bayangin batuk pilek dokternya gitu
4,habis dokternya
...,...
1494,dokternya disiruh kontrol tutup
1495,ok post dokternya cuti meledak
1497,tp gpp dokternya ganteng
1498,hdu berat ya gada dokternya kalo yg


Save the Data after doing Preprocessing

In [ ]:
data.to_excel('./dataset-training-preprocessing.xlsx', encoding='utf8', index=True)

**Mentranslate tweet_cleans menjadi bahasa inggris**

In [35]:
from google_trans_new import google_translator  
translator = google_translator()  

def convert_eng(tweet):
  return tweet

  translator.translate(tweet,lang_tgt='en')

data['tweet_english'] = data['tweet_cleans'].apply(convert_eng)
data.head(1000)

,tweet_cleans,tweet_english
0,dokternya ngasih gue obat ya gue keringetan mu...,dokternya ngasih gue obat ya gue keringetan mu...
1,oia yg dokternya tusuk ya,oia yg dokternya tusuk ya
2,dokternya,dokternya
3,bayangin batuk pilek dokternya gitu,bayangin batuk pilek dokternya gitu
4,habis dokternya,habis dokternya
...,...,...
1041,harian grgr kena cacar air gboleh mandi ama do...,harian grgr kena cacar air gboleh mandi ama do...
1042,wkwkwkkk ember emang lumrah kampusnya kampus s...,wkwkwkkk ember emang lumrah kampusnya kampus s...
1043,dokternya kebingungan,dokternya kebingungan
1044,ngga dibales pake beb sih dokternya,ngga dibales pake beb sih dokternya


In [37]:
import googletrans
from googletrans import *
translator = googletrans.Translator()

data['tweet_english'] = data['tweet_english'].astype(str) #changing datatype to string
data['tweet_english'] = data['tweet_cleans'].apply(translator.translate, src='auto', dest='en').apply(getattr, args=('text',))
data

,tweet_cleans,tweet_english
0,dokternya ngasih gue obat ya gue keringetan mu...,"the doctor gave me medicine, yes, I'm sweaty i..."
1,oia yg dokternya tusuk ya,oh the one the doctor stabbed ya
2,dokternya,the doctor
3,bayangin batuk pilek dokternya gitu,imagine the doctor's cough and cold
4,habis dokternya,finished the doctor
...,...,...
1494,dokternya disiruh kontrol tutup,the doctor was told to close the control
1495,ok post dokternya cuti meledak,ok the doctor's post exploded
1497,tp gpp dokternya ganteng,but it's okay the doctor is handsome
1498,hdu berat ya gada dokternya kalo yg,"it's heavy, there's no doctor for that"


**Melabeli Dataset Training dengan menggunakan TextBlob karna TextBlob support Eng**

In [38]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

In [46]:
ps = PorterStemmer() 

def stemming_data(x):
    return ps.stem(x)

data['tweet_english'] = data['tweet_english'].apply(stemming_data)

In [47]:
data_tweet = list(data['tweet_english'])
polaritas = 0

status = []
total_positif = total_negatif = total_netral = total = 0

for i, tweet in enumerate(data_tweet):
    analysis = TextBlob(tweet)
    polaritas += analysis.polarity

    if analysis.sentiment.polarity > 0.0:
        total_positif += 1
        status.append('Positif')
    elif analysis.sentiment.polarity == 0.0:
        total_netral += 1
        status.append('Netral')
    else:
        total_negatif += 1
        status.append('Negatif')

    total += 1 
    
print(f'Hasil Analisis Data:\nPositif = {total_positif}\nNetral = {total_netral}\nNegatif = {total_negatif}')
print(f'\nTotal Data : {total}')

Hasil Analisis Data:
Positif = 499
Netral = 510
Negatif = 377

Total Data : 1386


In [48]:
status = pd.DataFrame({'klasifikasi': status})
data['klasifikasi'] = status
data.tail()

,tweet_cleans,tweet_english,klasifikasi
1494,dokternya disiruh kontrol tutup,the doctor was told to close the control,NaN
1495,ok post dokternya cuti meledak,ok the doctor's post explod,NaN
1497,tp gpp dokternya ganteng,but it's okay the doctor is handsom,NaN
1498,hdu berat ya gada dokternya kalo yg,"it's heavy, there's no doctor for that",NaN
1499,dokternya rain kim bum kesurupan hantu jatuh h...,the doctor rain kim bum is possessed by a ghos...,NaN


In [49]:
data.to_excel('./dataset-training-preprocessing.xlsx', encoding='utf8', index=True)